# Developing/Testing

In [29]:
cwd = os.getcwd()
itop=cwd.find("cgshells/")+len("cgshells")
PROJECT_ROOT = cwd[:itop]

'/Users/kyle/Documents/Code/cgshells'

In [64]:
PROJECT_ROOT

PosixPath('/Users/kyle/Documents/Code/cgshells')

In [71]:
os.path.isdir('/Users/kyle/Documents/Code/cgshells')

True

In [66]:
'/Users/kyle/Documents/Code/cgshells'.replace('/','.')

'.Users.kyle.Documents.Code.cgshells'

In [68]:
".".join('/Users/kyle/Documents/Code/cgshells'.split('/'))

'.Users.kyle.Documents.Code.cgshells'

In [2]:
import numpy as np
import importlib
import os
import sys 
cwd = os.getcwd()
itop=cwd.find("cgshells/")+len("cgshells")
PROJECT_ROOT = cwd[:itop]
sys.path.insert(0, PROJECT_ROOT )    # needed for jupyter notebook

import utils.run_manager as rm
from utils.run_manager import PROJECT_ROOT, lmpunity, lmplocal
from utils.readsim import ReadSim

version = "v1"
curvsim = importlib.import_module(f"utils.curvsim.{version}")
Curvamer2D = rm.load_class(version, "curvamer2d", "Curvamer2D")
Curvamer3D = rm.load_class(version, "curvamer3d", "Curvamer3D")
versionpath = "/".join(curvsim.__name__.split("."))
DATASCRIPTS = f"{PROJECT_ROOT}/{versionpath}/DataScripts"

# import curvamer2d.Curvamer2D 
# from curvamer3d import Curvamer3D

In [42]:
versionpath

'utils/curvsim/v1'

In [41]:
curvsim.__name__

'utils.curvsim.v1'

In [40]:
curvsim.__name__.split(".")

['utils', 'curvsim', 'v1']

In [7]:
import pathlib

In [8]:
pathlib.Path(__file__).resolve().parents[1]

NameError: name '__file__' is not defined

In [16]:
versionpath = "/".join(curvsim.__name__.split("."))
f"{PROJECT_ROOT}/{versionpath}/DataScripts"

'/Users/kyle/Documents/Code/cgshells/utils/curvsim/v1/DataScripts'

In [2]:
sim = Curvamer2D()
sim.cwd

'/Users/kyle/Documents/Code/cgshells/reference'

In [2]:
sys.path

['..',
 '/Users/kyle/Documents/Code/cgshells/reference',
 '/opt/homebrew/anaconda3/lib/python39.zip',
 '/opt/homebrew/anaconda3/lib/python3.9',
 '/opt/homebrew/anaconda3/lib/python3.9/lib-dynload',
 '',
 '/Users/kyle/.local/lib/python3.9/site-packages',
 '/opt/homebrew/anaconda3/lib/python3.9/site-packages',
 '/opt/homebrew/anaconda3/lib/python3.9/site-packages/aeosa',
 '/opt/homebrew/anaconda3/lib/python3.9/site-packages/pymesh2-0.3-py3.9-macosx-11.1-arm64.egg']

In [10]:
import utils.run_manager as rm

In [3]:
PROJECT_ROOT

PosixPath('/Users/kyle/Documents/Code/cgshells')

In [31]:
print(PROJECT_ROOT)

/Users/kyle/Documents/Code/cgshells


In [14]:
INPUTS = rm.PROJECT_ROOT / "inputs"
print(INPUTS)

/Users/kyle/Documents/Code/cgshells/inputs


In [4]:
rm.lmpunity

'/home/kyltsullivan_umass_edu/lammps-23Jun2022/src/lmp_mpi'

In [5]:
rm.lmplocal

'/Users/kyle/Documents/Code/lammps-23Jun2022/src/lmp_mpi'

In [7]:
lmplocal

'/Users/kyle/Documents/Code/lammps-23Jun2022/src/lmp_mpi'

In [51]:
tlim_hrs = 0
tlim_min = 10

walltime = f'{tlim_hrs:02d}:{tlim_min:02d}:00'
walltime

'00:10:00'

In [52]:
##### PARTICLE #####
### Geometry
dimension = 2
dcore = 1.0    # hard core diameter of beads (dcore approx thickness of one DNA helix 3.5nm)
t0 = 1.0 * dcore    # structural thickness
wx = 56 * dcore    # curvamer width (arclength along midline)
r0 = 34 * dcore
Nbeads = 90    # number of beads per layer (2Nbeads is beads per curvamer)
fraction = 1/3    # middle patch of beads has width = fraction * wx

if r0 == "flat":
    k_0 = 0
else:
    k_0 = 1/r0

### Elasticity
kh = 10000
nu = 0.3
d = wx/(Nbeads-1)   # bead spacing
kvkh = 1 #(12 * t0**2 * (1-nu)) / (3 * d**2 * (1-nu) - 4 * t0**2)
kckh = (3 * (d**2 + t0**2) * (1-nu))/(2 * t0**2)

### Interactions
pair_ints = "repulsive"
sigma = 1*dcore
epsilon = 0.012938329803772514
shift = dcore - 2**(1/6)*sigma     # shift factor to make sure lj minimum is at dcore
ljcut = 5*sigma #t0 + 2*dcore               # cutoff distance for attractive lj potential
wcacut = dcore    # cutoff distance for repulsive wca potential
# softsigma = 1*sigma
# softepsilon = 1 * epsilon
softsigma = 5*sigma
softepsilon = 5e-8 * epsilon
softshift = 0 #softcore - 2**(1/6)*softsigma
softcut = 2**(1/6) * softsigma

##### SIMULATION #####
config = "dispersed"
simtype = "md"


### Concentration
nshells = 256
phi = 0.01    # concentration of molecules (area fraction) - only for MD
Nx = 4    # number of particle columns for initial config (applies if simpath_old = "none")
Ny = int(nshells/Nx)

### Dynamics/Minimization Settings
simtype = "md"
datagz = True
trajgz = False
screen = True    # output lammps log to screen
minstyle = "cg"
etol = 1e-14
maxiter = 100000
Tstart = 1.0
Tstop = Tstart
Tdamp = 10
seed = 15298
timestep = 0.0005
runsteps = 10000
dumpfreq = 100 # maxiter
thermofreq = 100
# force  = 300
px = 2
py = 2
pz = 1

### Simulation Box
v0 = wx * (t0 + dcore)
lbox = np.sqrt(nshells * v0 / phi)    # side length of (square) sim box to give proper concentration
xlo = -lbox/2
xhi = lbox/2
ylo = -lbox/2
yhi = lbox/2
zlo = -0.5
zhi = 0.5

### Simulation Directories
load_simpath = "none" # location of trajectory file to load in ("none" will create initial melt lattice)
simpath = "test" #"InitialConfigs/Nbeads-{}-Nmols-{}/phi-{:.3f}".format(N,Nmols,phi)

### Computation
# computer = "local"
computer = "unity"
nnodes = 1
mem = 16 #GB
tlim_hrs = 0
tlim_min = 10
partition = "cpu-preempt"
jobname = os.path.splitext(os.path.basename(sys.argv[0]))[0]
requested_walltime = f'{tlim_hrs:02d}:{tlim_min:02d}:00'

In [60]:
xhi

846.640419540669

In [59]:
type(xhi)

numpy.float64

In [62]:
type(float(xhi))

float

In [76]:
khlist = [5,10,15]
print(f"khlist = {khlist}")

khlist = [5, 10, 15]


In [81]:
import datetime
import pytz

In [86]:
current_time = datetime.datetime.now(pytz.timezone("America/New_York"))

In [88]:
print(current_time)

2025-09-11 22:20:36.588241-04:00


In [9]:
meta = rm.read_metadata(PROJECT_ROOT / "reference")
meta['logistics']['run_counter']

1

In [8]:
rm.update_metadata(PROJECT_ROOT / "reference",increment_run = True)
meta = rm.read_metadata(PROJECT_ROOT / "reference")
meta['logistics']['run_counter']

1

In [10]:
rm.update_metadata(PROJECT_ROOT / "reference",sub = 'logistics',params = {'datatime':65})
meta = rm.read_metadata(PROJECT_ROOT / "reference")
meta['logistics']

{'computer': 'unity',
 'jobname': 'ipykernel_launcher',
 'simpath': 'test',
 'load_simpath': 'none',
 'nnodes': 1,
 'cpus': 4,
 'mem': 16,
 'partition': 'cpu-preempt',
 'requested_walltime': '00:10:00',
 'run_counter': 1,
 'datatime': 65}

In [12]:
rm.update_metadata(PROJECT_ROOT / "reference",params = {'note':'test'})
meta = rm.read_metadata(PROJECT_ROOT / "reference")
meta['note']

'test'

In [13]:
meta

{'particle': {'geometry': {'dimension': 2,
   'dcore': 1.0,
   't0': 1.0,
   'wx': 56.0,
   'r0': 34.0,
   'Nbeads': 90,
   'fraction': 0.3333333333333333},
  'elasticity': {'nu': 0.3,
   'kh': 10000,
   'kckh': 1.4657050877414468,
   'kvkh': 1},
  'interactions': {'sigma': 1.0,
   'epsilon': 0.012938329803772514,
   'shift': -0.12246204830937302,
   'ljcut': 5.0,
   'wcacut': 1.0,
   'softsigma': 5.0,
   'softepsilon': 6.469164901886257e-10,
   'softshift': 0,
   'softcut': 5.612310241546865,
   'pair_ints': 'repulsive'}},
 'simulation': {'simtype': 'md',
  'config': 'dispersed',
  'nshells': 256,
  'phi': 0.01,
  'xlo': -846.640419540669,
  'xhi': 846.640419540669,
  'ylo': -846.640419540669,
  'yhi': 846.640419540669,
  'zlo': -0.5,
  'zhi': 0.5,
  'simbox_x': 1693.280839081338,
  'simbox_y': 1693.280839081338,
  'simbox_z': 1.0,
  'thermofreq': 100,
  'dumpfreq': 100,
  'Tstart': 1.0,
  'Tstop': 1.0,
  'Tdamp': 10,
  'seed': 15298,
  'timestep': 0.0005,
  'runsteps': 10000,
  'mins

In [16]:
  Stopping criterion = walltime limit reached

In [45]:
np.arange(5).tolist()

[0, 1, 2, 3, 4]

In [46]:
list(np.arange(5))

[0, 1, 2, 3, 4]

In [47]:
list([0,2,4])

[0, 2, 4]

In [51]:
int('12\n')*2

24

In [69]:
3//2

1

In [65]:
int(np.ceil(11/2))

6

In [77]:
nstack = 7
top = int(np.ceil(nstack/2))+1
if nstack % 2 == 0:
    bot = int(np.ceil(nstack/2))
else: 
    bot = int(np.ceil(nstack/2))-1
print(f"bottom half = 1 ... {bot}")
print(f"bottom half = {top} ... {nstack}")

bottom half = 1 ... 3
bottom half = 5 ... 7
